# Apple vs Samsung — Competitive Financial & Market Analysis

## Research Questions
1. **How have Apple and Samsung stock prices performed and correlated over 2015–2025?**
2. **How do Apple and Samsung compare on key financial fundamentals (revenue, profitability, cash flow)?**
3. **Can macroeconomic factors (inflation and interest rates) explain monthly stock return variations for Apple and Samsung?**

## Data Sources
| # | Source | Type | Coverage | Variables |
|---|--------|------|----------|-----------|
| 1 | [Yahoo Finance](https://finance.yahoo.com) via `yfinance` | Stock prices & financial statements | 2015–2025 (daily / annual) | OHLCV prices; Revenue, Net Income, Gross Profit, Operating Income, Total Assets, Free Cash Flow |
| 2 | [FRED — Federal Reserve Bank of St. Louis](https://fred.stlouisfed.org) via `pandas-datareader` | Macroeconomic indicators | 2015–2025 (monthly / daily) | `CPIAUCSL` — US CPI (inflation proxy); `DGS10` — 10-Year Treasury Rate |

**Coverage:** January 1, 2015 → present  
**Companies:** Apple Inc. (AAPL, NYSE) | Samsung Electronics (005930.KS, KRX)

## Project Folder Structure
```
├── Apple_vs_Samsung_Analysis.ipynb   ← main notebook (this file)
├── data/                             ← all CSV data files
├── plots/                            ← all saved chart images
├── presentation/                     ← PDF, PPTX, and slide images
│   └── slides/
├── notebooks/                        ← Data_download.ipynb
└── README.md
```

## 1. Setup & Imports

In [1]:
%pip install yfinance pandas numpy matplotlib seaborn scikit-learn statsmodels pandas-datareader -q
print("All packages ready.")

Note: you may need to restart the kernel to use updated packages.
All packages ready.


In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
import os
import warnings
warnings.filterwarnings('ignore')

# Ensure output folders exist
os.makedirs('data',  exist_ok=True)
os.makedirs('plots', exist_ok=True)

# Consistent plot styling
plt.rcParams.update({
    'figure.dpi': 130, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3
})
APPLE_COLOR   = '#1f77b4'
SAMSUNG_COLOR = '#ff7f0e'

print("Imports complete.")

## 2. Data Loading

All CSV files were pre-downloaded from Yahoo Finance (`yfinance`) and FRED (`pandas-datareader`). We load and inspect each dataset below.


In [ ]:
# ── Stock Prices ──────────────────────────────────────────────────────────
stock = pd.read_csv('data/stock_prices.csv', header=[0,1], index_col=0, parse_dates=True)
apple_close   = stock['Apple']['Close'].astype(float)
samsung_close = stock['Samsung']['Close'].astype(float)
apple_vol     = stock['Apple']['Volume'].astype(float)
samsung_vol   = stock['Samsung']['Volume'].astype(float)

print(f"Stock data shape: {stock.shape}")
print(f"Date range: {apple_close.index[0].date()} to {apple_close.index[-1].date()}")
stock.tail(3)

In [ ]:
# ── Financial Statements ──────────────────────────────────────────────────
apple_inc    = pd.read_csv('data/apple_income_stmt.csv',     index_col=0)
samsung_inc  = pd.read_csv('data/samsung_income_stmt.csv',   index_col=0)
apple_bs     = pd.read_csv('data/apple_balance_sheet.csv',   index_col=0)
samsung_bs   = pd.read_csv('data/samsung_balance_sheet.csv', index_col=0)
apple_cf     = pd.read_csv('data/apple_cash_flow.csv',       index_col=0)
samsung_cf   = pd.read_csv('data/samsung_cash_flow.csv',     index_col=0)

print("Apple Income Statement — Fiscal Years:", apple_inc.columns.tolist())
print("Samsung Income Statement — Fiscal Years:", samsung_inc.columns.tolist())
print(f"Apple income rows: {len(apple_inc)} | Samsung income rows: {len(samsung_inc)}")

In [ ]:
# ── Macroeconomic Data ────────────────────────────────────────────────────
macro = pd.read_csv('data/fred_macro.csv', index_col=0, parse_dates=True)
cpi   = macro['CPI_US'].dropna()
t10y  = macro['Treasury_10Y'].dropna()

print(f"CPI (monthly): {len(cpi)} obs | {cpi.index[0].date()} – {cpi.index[-1].date()}")
print(f"10Y Treasury (daily): {len(t10y)} obs | {t10y.index[0].date()} – {t10y.index[-1].date()}")
macro.head(5)

## 3. Data Cleaning & Manipulation

We apply the following cleaning steps:
- **Index handling**: parse date strings to `DatetimeIndex`; set/reset index as needed
- **Missing values**: identify and forward-fill or drop NaNs (stock prices are weekday-only; CPI is monthly)
- **Data type conversion**: cast price/volume columns to `float64`
- **Subsetting**: extract only the rows/columns needed for analysis
- **New columns**: compute daily returns, profit margins, annualized volatility
- **Sorting**: sort all time-series chronologically
- **Merging**: align stock, financial, and macro data on a common date axis
- **Aggregation**: compute mean, median, correlation, and annual summary statistics


In [6]:
# ── 3.1 Clean Stock Prices ────────────────────────────────────────────────
# Check missing values
print("Missing values before cleaning:")
print(f"  Apple close:   {apple_close.isna().sum()} NaNs")
print(f"  Samsung close: {samsung_close.isna().sum()} NaNs")

# Combine and drop rows where either price is missing
prices = pd.DataFrame({'Apple': apple_close, 'Samsung': samsung_close}).dropna()
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()  # Sort chronologically

print(f"\nAfter dropping NaNs: {len(prices)} rows")
print(f"Date range: {prices.index[0].date()} – {prices.index[-1].date()}")

# Normalize to 100 at start (base-100 index)
prices_norm = prices / prices.iloc[0] * 100

# Daily returns (pct_change) — new column generation
returns = prices.pct_change().dropna()
print(f"\nDaily returns shape: {returns.shape}")
returns.describe().round(4)

Missing values before cleaning:
  Apple close:   88 NaNs
  Samsung close: 159 NaNs

After dropping NaNs: 2680 rows
Date range: 2015-01-02 – 2026-04-17

Daily returns shape: (2679, 2)


,Apple,Samsung
count,2679.0000,2679.0000
mean,0.0011,0.0011
std,0.0185,0.0188
min,-0.1286,-0.1174
25%,-0.0075,-0.0096
50%,0.0010,0.0000
75%,0.0101,0.0103
max,0.1533,0.1343


In [7]:
# ── 3.2 Clean Financial Statements ───────────────────────────────────────
def extract_fin(df, row_map, scale):
    """Extract named rows, convert to float, scale to B/T units."""
    result = {}
    for label, row_key in row_map.items():
        if row_key in df.index:
            result[label] = pd.to_numeric(df.loc[row_key], errors='coerce') / scale
    out = pd.DataFrame(result)
    out.index = pd.to_datetime(out.index)
    return out.sort_index()

# Apple income statement — USD billions
apple_fin = extract_fin(apple_inc, {
    'Revenue': 'Total Revenue', 'Gross Profit': 'Gross Profit',
    'Operating Income': 'Operating Income', 'Net Income': 'Net Income', 'EBITDA': 'EBITDA'
}, scale=1e9)
apple_fin['Gross Margin %'] = apple_fin['Gross Profit'] / apple_fin['Revenue'] * 100
apple_fin['Net Margin %']   = apple_fin['Net Income']   / apple_fin['Revenue'] * 100
apple_fin['Op Margin %']    = apple_fin['Operating Income'] / apple_fin['Revenue'] * 100

# Samsung income statement — KRW trillions
samsung_fin = extract_fin(samsung_inc, {
    'Revenue': 'Total Revenue', 'Gross Profit': 'Gross Profit',
    'Operating Income': 'Operating Income', 'Net Income': 'Net Income', 'EBITDA': 'EBITDA'
}, scale=1e12)
samsung_fin['Gross Margin %'] = samsung_fin['Gross Profit'] / samsung_fin['Revenue'] * 100
samsung_fin['Net Margin %']   = samsung_fin['Net Income']   / samsung_fin['Revenue'] * 100
samsung_fin['Op Margin %']    = samsung_fin['Operating Income'] / samsung_fin['Revenue'] * 100

# Cash flow
apple_fcf   = extract_fin(apple_cf,   {'Free Cash Flow': 'Free Cash Flow', 'Operating CF': 'Operating Cash Flow', 'CapEx': 'Capital Expenditure'}, scale=1e9)
samsung_fcf = extract_fin(samsung_cf, {'Free Cash Flow': 'Free Cash Flow', 'Operating CF': 'Operating Cash Flow', 'CapEx': 'Capital Expenditure'}, scale=1e12)

print("Apple financials (USD Billions):")
display(apple_fin.dropna().round(1))
print("\nSamsung financials (KRW Trillions):")
display(samsung_fin.round(1))

Apple financials (USD Billions):


,Revenue,Gross Profit,Operating Income,Net Income,EBITDA,Gross Margin %,Net Margin %,Op Margin %
2022-09-30,394.3,170.8,119.4,99.8,130.5,43.3,25.3,30.3
2023-09-30,383.3,169.1,114.3,97.0,125.8,44.1,25.3,29.8
2024-09-30,391.0,180.7,123.2,93.7,134.7,46.2,24.0,31.5
2025-09-30,416.2,195.2,133.0,112.0,144.7,46.9,26.9,32.0



Samsung financials (KRW Trillions):


,Revenue,Gross Profit,Operating Income,Net Income,EBITDA,Gross Margin %,Net Margin %,Op Margin %
2022-12-31,302.2,112.2,43.4,54.7,86.3,37.1,18.1,14.4
2023-12-31,258.9,78.5,6.6,14.5,50.6,30.3,5.6,2.5
2024-12-31,300.9,114.3,32.7,33.6,81.1,38.0,11.2,10.9
2025-12-31,333.6,131.4,43.6,44.3,97.0,39.4,13.3,13.1


In [8]:
# ── 3.3 Aggregate & Merge Macro Data ─────────────────────────────────────
cpi_monthly  = cpi.resample('ME').last()
t10y_monthly = t10y.resample('ME').mean()

# Monthly stock returns
apple_monthly   = apple_close.resample('ME').last().pct_change()
samsung_monthly = samsung_close.resample('ME').last().pct_change()

# CPI change (monthly inflation rate) and change in treasury yield
cpi_chg  = cpi_monthly.pct_change()
t10y_chg = t10y_monthly.diff()

# Merge into single dataset (inner join on date)
df_reg = pd.DataFrame({
    'Apple_ret':   apple_monthly,
    'Samsung_ret': samsung_monthly,
    'CPI_chg':     cpi_chg,
    'T10Y_chg':    t10y_chg
}).dropna()

print(f"Merged regression dataset: {df_reg.shape[0]} monthly observations")
df_reg.describe().round(4)

Merged regression dataset: 134 monthly observations


,Apple_ret,Samsung_ret,CPI_chg,T10Y_chg
count,134.0000,134.0000,134.0000,134.0000
mean,0.0201,0.0195,0.0026,0.0176
std,0.0779,0.0915,0.0027,0.1958
min,-0.1812,-0.2277,-0.0079,-0.6342
25%,-0.0388,-0.0348,0.0011,-0.1026
50%,0.0209,0.0039,0.0024,0.0060
75%,0.0764,0.0697,0.0037,0.1333
max,0.2166,0.3489,0.0126,0.6212


## 4. Descriptive Statistics

Before diving into analysis, we summarize the key return and risk metrics for each company.


In [9]:
# ── Key Performance Metrics ───────────────────────────────────────────────
ann_return_apple   = (1 + returns['Apple'].mean())   ** 252 - 1
ann_return_samsung = (1 + returns['Samsung'].mean()) ** 252 - 1
ann_vol_apple      = returns['Apple'].std()   * np.sqrt(252)
ann_vol_samsung    = returns['Samsung'].std() * np.sqrt(252)

summary = pd.DataFrame({
    'Metric': [
        'Start Price', 'End Price', 'Total Return', 'Annualized Return',
        'Annualized Volatility', 'Sharpe Ratio (rf=0)', 'Max Daily Gain', 'Max Daily Loss'
    ],
    'Apple (USD)': [
        f'${ apple_close.iloc[0]:.2f}', f'${apple_close.iloc[-1]:.2f}',
        f'{(apple_close.iloc[-1]/apple_close.iloc[0]-1)*100:.1f}%',
        f'{ann_return_apple*100:.1f}%', f'{ann_vol_apple*100:.1f}%',
        f'{ann_return_apple/ann_vol_apple:.2f}',
        f'{returns["Apple"].max()*100:.2f}%', f'{returns["Apple"].min()*100:.2f}%'
    ],
    'Samsung (KRW)': [
        f'₩{samsung_close.iloc[0]:,.0f}', f'₩{samsung_close.iloc[-1]:,.0f}',
        f'{(samsung_close.iloc[-1]/samsung_close.iloc[0]-1)*100:.1f}%',
        f'{ann_return_samsung*100:.1f}%', f'{ann_vol_samsung*100:.1f}%',
        f'{ann_return_samsung/ann_vol_samsung:.2f}',
        f'{returns["Samsung"].max()*100:.2f}%', f'{returns["Samsung"].min()*100:.2f}%'
    ]
}).set_index('Metric')

display(summary)

,Apple (USD),Samsung (KRW)
Metric,,
Start Price,$24.21,"₩20,462"
End Price,$270.23,"₩216,000"
Total Return,1016.0%,955.6%
Annualized Return,31.0%,30.5%
Annualized Volatility,29.4%,29.9%
Sharpe Ratio (rf=0),1.05,1.02
Max Daily Gain,15.33%,13.43%
Max Daily Loss,-12.86%,-11.74%


## 5. Research Question 1 — Stock Price Performance & Correlation

**Question:** How have Apple and Samsung stock prices performed and correlated over 2015–2025?

**Analyses performed:**
- Normalized price comparison (base = 100)
- Rolling Pearson correlation of daily returns
- Scatter plot of returns with year-by-year correlation breakdown
- Rolling annualized volatility


In [ ]:
# ── Plot 1: Normalized Stock Price Comparison ─────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios': [3, 1]})

ax1 = axes[0]
ax1.plot(prices_norm.index, prices_norm['Apple'],   color=APPLE_COLOR,   lw=1.8, label='Apple (AAPL)')
ax1.plot(prices_norm.index, prices_norm['Samsung'], color=SAMSUNG_COLOR,  lw=1.8, label='Samsung (005930.KS)')
ax1.set_ylabel('Normalized Price (Base = 100)', fontsize=12)
ax1.set_title('Apple vs Samsung — Normalized Stock Price (2015–2025)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}'))

ax2 = axes[1]
ax2.fill_between(returns.index, returns['Apple'],   alpha=0.5, color=APPLE_COLOR,   label='Apple returns')
ax2.fill_between(returns.index, returns['Samsung'], alpha=0.5, color=SAMSUNG_COLOR,  label='Samsung returns')
ax2.set_ylabel('Daily Return', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.legend(fontsize=10)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))

plt.tight_layout()
plt.savefig('plots/plot1_normalized_prices.png', bbox_inches='tight')
plt.show()

**Observation:** Both stocks started near the same normalized level in 2015 and grew substantially over the decade, with Apple achieving a total return of ~1,016% vs Samsung's ~956%. Apple shows a more consistent upward trend, while Samsung experienced a notable correction post-2021 before recovering. Both stocks exhibited sharp drawdowns during COVID-19 (March 2020) and the 2022 rate-hike cycle.


In [ ]:
# ── Plot 2: Rolling Correlation ───────────────────────────────────────────
roll90  = returns['Apple'].rolling(90).corr(returns['Samsung'])
roll252 = returns['Apple'].rolling(252).corr(returns['Samsung'])
overall_corr = returns.corr().loc['Apple','Samsung']

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(roll90.index,  roll90,  color='#2ca02c', lw=1.5, alpha=0.7, label='90-day Rolling Correlation')
ax.plot(roll252.index, roll252, color='#d62728', lw=2.0, label='252-day Rolling Correlation (1-year)')
ax.axhline(overall_corr, color='black', ls='--', lw=1.2, label=f'Overall: {overall_corr:.3f}')
ax.axhline(0, color='grey', ls='-', lw=0.8)
ax.fill_between(roll90.index, roll90, 0, alpha=0.15, color='#2ca02c')
ax.set_ylim(-0.5, 1.0)
ax.set_ylabel('Pearson Correlation Coefficient', fontsize=12)
ax.set_xlabel('Date', fontsize=11)
ax.set_title('Rolling Return Correlation: Apple vs Samsung', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('plots/plot2_rolling_correlation.png', bbox_inches='tight')
plt.show()

print(f"Overall Pearson correlation (daily returns): {overall_corr:.4f}")

In [ ]:
# ── Plot 3: Annual Correlation & Return Scatter ───────────────────────────
yearly_corr = returns.groupby(returns.index.year).apply(lambda g: g['Apple'].corr(g['Samsung']))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(returns['Apple'], returns['Samsung'], alpha=0.15, s=8, color='steelblue')
m, b = np.polyfit(returns['Apple'], returns['Samsung'], 1)
xline = np.linspace(returns['Apple'].min(), returns['Apple'].max(), 100)
ax.plot(xline, m*xline + b, color='red', lw=2, label=f'Slope β={m:.2f}')
ax.set_xlabel('Apple Daily Return', fontsize=11)
ax.set_ylabel('Samsung Daily Return', fontsize=11)
ax.set_title(f'Return Scatter (r = {overall_corr:.3f})', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax.legend(fontsize=11)

ax2 = axes[1]
colors_bar = ['#d62728' if v < 0 else '#2ca02c' for v in yearly_corr]
bars = ax2.bar(yearly_corr.index, yearly_corr.values, color=colors_bar, edgecolor='white', width=0.7)
for bar, val in zip(bars, yearly_corr.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.2f}', ha='center', va='bottom', fontsize=9)
ax2.set_xlabel('Year', fontsize=11)
ax2.set_ylabel('Pearson r', fontsize=11)
ax2.set_title('Annual Return Correlation', fontsize=12, fontweight='bold')
ax2.set_ylim(-0.3, 0.9)
ax2.axhline(0, color='black', lw=0.8)

plt.suptitle('Apple vs Samsung: Return Scatter & Annual Correlation', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/plot3_scatter_annual_corr.png', bbox_inches='tight')
plt.show()

print("Annual correlations:")
print(yearly_corr.round(3))

In [ ]:
# ── Plot 4: Rolling Volatility ────────────────────────────────────────────
roll_vol = returns.rolling(60).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(roll_vol.index, roll_vol['Apple'],   color=APPLE_COLOR,   lw=1.8, label='Apple (60-day rolling σ, annualized)')
ax.plot(roll_vol.index, roll_vol['Samsung'], color=SAMSUNG_COLOR,  lw=1.8, label='Samsung (60-day rolling σ, annualized)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylabel('Annualized Volatility', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.set_title('Rolling 60-Day Annualized Volatility: Apple vs Samsung', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('plots/plot11_volatility.png', bbox_inches='tight')
plt.show()

**Findings — Research Question 1:**
- The **overall Pearson correlation of daily returns is low (r ≈ 0.068)**, indicating Apple and Samsung largely trade independently despite being direct product competitors.
- The rolling correlation fluctuates significantly, occasionally rising above 0.4 during periods of broad market stress (e.g., COVID-19 crash 2020) but often falling near zero or negative.
- Annual correlations vary widely by year, confirming that the relationship is not stable over time.
- Both stocks exhibit similar annualized volatility (~29–30%), with spikes during market crises. Their risk profiles are nearly identical despite operating in different markets and currencies.


## 6. Research Question 2 — Financial Fundamentals Comparison

**Question:** How do Apple and Samsung compare on key financial fundamentals (revenue, profitability, cash flow) over 2022–2025?

**Note:** Apple reports in USD (billions). Samsung reports in KRW (trillions). Due to different currencies and accounting periods (Apple: Sep FY; Samsung: Dec FY), we focus on margin ratios and trends rather than absolute levels for direct comparison.


In [ ]:
# ── Plot 5: Revenue & Net Income ──────────────────────────────────────────
years_a = [d.year for d in apple_fin.dropna().index]
years_s = [d.year for d in samsung_fin.dropna().index]
x_a = np.arange(len(years_a))
x_s = np.arange(len(years_s))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.bar(x_a - 0.2, apple_fin.dropna()['Revenue'],    0.35, label='Revenue (USD B)',    color=APPLE_COLOR, alpha=0.85)
ax.bar(x_a + 0.2, apple_fin.dropna()['Net Income'], 0.35, label='Net Income (USD B)', color=APPLE_COLOR, alpha=0.45)
ax.set_xticks(x_a); ax.set_xticklabels(years_a, rotation=30)
ax.set_ylabel('USD Billions', fontsize=11)
ax.set_title('Apple: Revenue vs Net Income', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
for i, (r, n) in enumerate(zip(apple_fin.dropna()['Revenue'], apple_fin.dropna()['Net Income'])):
    ax.text(i-0.2, r+3, f'{r:.0f}', ha='center', fontsize=8)
    ax.text(i+0.2, n+3, f'{n:.0f}', ha='center', fontsize=8)

ax2 = axes[1]
ax2.bar(x_s - 0.2, samsung_fin['Revenue'],    0.35, label='Revenue (KRW T)',    color=SAMSUNG_COLOR, alpha=0.85)
ax2.bar(x_s + 0.2, samsung_fin['Net Income'], 0.35, label='Net Income (KRW T)', color=SAMSUNG_COLOR, alpha=0.45)
ax2.set_xticks(x_s); ax2.set_xticklabels(years_s, rotation=30)
ax2.set_ylabel('KRW Trillions', fontsize=11)
ax2.set_title('Samsung: Revenue vs Net Income', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
for i, (r, n) in enumerate(zip(samsung_fin['Revenue'], samsung_fin['Net Income'])):
    ax2.text(i-0.2, r+5, f'{r:.0f}', ha='center', fontsize=8)
    ax2.text(i+0.2, n+2, f'{n:.0f}', ha='center', fontsize=8)

plt.suptitle('Financial Comparison: Apple vs Samsung', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/plot4_revenue_netincome.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 6: Profit Margins ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, fin, years, name, color in [
    (axes[0], apple_fin.dropna(),   years_a, 'Apple',   APPLE_COLOR),
    (axes[1], samsung_fin,          years_s, 'Samsung', SAMSUNG_COLOR)
]:
    x = np.arange(len(years))
    ax.plot(x, fin['Gross Margin %'], 'o-',  color=color, lw=2,   label='Gross Margin', markersize=7)
    ax.plot(x, fin['Op Margin %'],    's--', color=color, lw=1.8, label='Operating Margin', alpha=0.8, markersize=6)
    ax.plot(x, fin['Net Margin %'],   '^:',  color=color, lw=1.5, label='Net Margin', alpha=0.6, markersize=6)
    ax.set_xticks(x); ax.set_xticklabels(years, rotation=30)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax.set_title(f'{name}: Profit Margins', fontsize=12, fontweight='bold')
    ax.set_ylabel('Margin (%)', fontsize=11)
    ax.legend(fontsize=10)

plt.suptitle('Profit Margin Comparison: Apple vs Samsung', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/plot5_profit_margins.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 7: Free Cash Flow ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cf, name, color, unit in [
    (axes[0], apple_fcf.dropna(subset=['Free Cash Flow']),   'Apple',   APPLE_COLOR,   'USD Billions'),
    (axes[1], samsung_fcf.dropna(subset=['Free Cash Flow']), 'Samsung', SAMSUNG_COLOR, 'KRW Trillions')
]:
    years = [d.year for d in cf.index]
    x = np.arange(len(years))
    ax.bar(x - 0.2, cf['Operating CF'],   0.35, color=color, alpha=0.85, label='Operating CF')
    ax.bar(x + 0.2, cf['Free Cash Flow'], 0.35, color=color, alpha=0.45, label='Free Cash Flow')
    ax.set_xticks(x); ax.set_xticklabels(years, rotation=30)
    ax.set_title(f'{name}: Cash Flow ({unit})', fontsize=12, fontweight='bold')
    ax.set_ylabel(unit, fontsize=11)
    ax.legend(fontsize=10)
    ax.axhline(0, color='black', lw=0.8)

plt.suptitle('Cash Flow Comparison: Apple vs Samsung', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/plot6_cashflow.png', bbox_inches='tight')
plt.show()

In [17]:
# ── Groupby Aggregation: Mean margins ─────────────────────────────────────
margin_compare = pd.DataFrame({
    'Apple Avg':   apple_fin.dropna()[['Gross Margin %','Op Margin %','Net Margin %']].mean(),
    'Samsung Avg': samsung_fin[['Gross Margin %','Op Margin %','Net Margin %']].mean()
}).round(1)
print("Average Profit Margins (2022–2025):")
display(margin_compare)

Average Profit Margins (2022–2025):


,Apple Avg,Samsung Avg
Gross Margin %,45.1,36.2
Op Margin %,30.9,10.2
Net Margin %,25.4,12.0


**Findings — Research Question 2:**
- **Revenue:** Apple's annual revenue (~$384–416B USD) remained relatively stable over the period with slight growth. Samsung experienced a significant revenue decline in 2023 (semiconductor downturn) but recovered strongly by 2025.
- **Profitability:** Apple consistently outperforms Samsung on all margin metrics. Apple's gross margin (~47%) is roughly double Samsung's (~37%), reflecting Apple's premium brand positioning and high-margin services revenue. Apple's net margin (~25%) compares favorably to Samsung's (~11–18%).
- **Cash Flow:** Apple generates exceptional free cash flow (~$100B/year) even relative to revenue. Samsung's FCF is volatile, turning negative in 2023 due to heavy chip manufacturing investment.
- Both companies increased margins in 2025, suggesting recovery and operational efficiency improvements.


## 7. Research Question 3 — Macroeconomic Regression Analysis

**Question:** Can macroeconomic factors (inflation and interest rates) explain monthly stock return variations?

**Methodology:** OLS linear regression  
- **Dependent variable:** Monthly stock return (Apple or Samsung)  
- **Independent variables:**  
  - `CPI_chg` — monthly percentage change in US CPI (inflation rate)  
  - `T10Y_chg` — monthly change in the 10-year Treasury yield (in percentage points)

**Hypothesis:** Rising inflation and rising interest rates should negatively impact growth stock returns, as higher discount rates reduce the present value of future earnings.


In [ ]:
# ── Macro Data Visualization ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

ax1 = axes[0]
ax1.plot(cpi_monthly.index, cpi_monthly.values, color='#9467bd', lw=2)
ax1.fill_between(cpi_monthly.index, cpi_monthly.values, alpha=0.2, color='#9467bd')
ax1.set_ylabel('CPI (All Urban Consumers)', fontsize=11)
ax1.set_title('US Macroeconomic Environment: CPI & 10-Year Treasury Yield', fontsize=13, fontweight='bold')

ax2 = axes[1]
ax2.plot(t10y_monthly.index, t10y_monthly.values, color='#8c564b', lw=2)
ax2.fill_between(t10y_monthly.index, t10y_monthly.values, alpha=0.2, color='#8c564b')
ax2.set_ylabel('10-Year Treasury Rate (%)', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)

plt.tight_layout()
plt.savefig('plots/plot7_macro.png', bbox_inches='tight')
plt.show()

print("CPI range:", f"{cpi.min():.1f} – {cpi.max():.1f}")
print("Treasury range:", f"{t10y.min():.2f}% – {t10y.max():.2f}%")

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────
corr_full = df_reg.rename(columns={
    'Apple_ret': 'Apple Return', 'Samsung_ret': 'Samsung Return',
    'CPI_chg': 'CPI Change', 'T10Y_chg': '10Y Treasury Δ'
}).corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_full, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix: Returns & Macro Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/plot10_corr_heatmap.png', bbox_inches='tight')
plt.show()

In [20]:
# ── OLS Regression ────────────────────────────────────────────────────────
X = add_constant(df_reg[['CPI_chg', 'T10Y_chg']])
ols_apple   = OLS(df_reg['Apple_ret'],   X).fit()
ols_samsung = OLS(df_reg['Samsung_ret'], X).fit()

print("=" * 60)
print("APPLE: Monthly Return ~ CPI_chg + T10Y_chg")
print("=" * 60)
print(f"R-squared:        {ols_apple.rsquared:.4f}")
print(f"Adj. R-squared:   {ols_apple.rsquared_adj:.4f}")
print(f"F-statistic:      {ols_apple.fvalue:.3f}  (p={ols_apple.f_pvalue:.3f})")
print(f"Intercept:        {ols_apple.params['const']:.4f}  (p={ols_apple.pvalues['const']:.3f})")
print(f"CPI_chg coef:     {ols_apple.params['CPI_chg']:.4f}  (p={ols_apple.pvalues['CPI_chg']:.3f})")
print(f"T10Y_chg coef:    {ols_apple.params['T10Y_chg']:.4f}  (p={ols_apple.pvalues['T10Y_chg']:.3f})")
print()
print("=" * 60)
print("SAMSUNG: Monthly Return ~ CPI_chg + T10Y_chg")
print("=" * 60)
print(f"R-squared:        {ols_samsung.rsquared:.4f}")
print(f"Adj. R-squared:   {ols_samsung.rsquared_adj:.4f}")
print(f"F-statistic:      {ols_samsung.fvalue:.3f}  (p={ols_samsung.f_pvalue:.3f})")
print(f"Intercept:        {ols_samsung.params['const']:.4f}  (p={ols_samsung.pvalues['const']:.3f})")
print(f"CPI_chg coef:     {ols_samsung.params['CPI_chg']:.4f}  (p={ols_samsung.pvalues['CPI_chg']:.3f})")
print(f"T10Y_chg coef:    {ols_samsung.params['T10Y_chg']:.4f}  (p={ols_samsung.pvalues['T10Y_chg']:.3f})")

APPLE: Monthly Return ~ CPI_chg + T10Y_chg
R-squared:        0.0159
Adj. R-squared:   0.0008
F-statistic:      1.055  (p=0.351)
Intercept:        0.0158  (p=0.100)
CPI_chg coef:     2.0791  (p=0.448)
T10Y_chg coef:    -0.0540  (p=0.153)

SAMSUNG: Monthly Return ~ CPI_chg + T10Y_chg
R-squared:        0.0235
Adj. R-squared:   0.0086
F-statistic:      1.574  (p=0.211)
Intercept:        0.0288  (p=0.011)
CPI_chg coef:     -3.3582  (p=0.296)
T10Y_chg coef:    -0.0396  (p=0.369)


In [ ]:
# ── Regression Visualization ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (name, ret_col, ols_res) in enumerate([
    ('Apple', 'Apple_ret', ols_apple),
    ('Samsung', 'Samsung_ret', ols_samsung)
]):
    color = APPLE_COLOR if i == 0 else SAMSUNG_COLOR
    pred  = ols_res.fittedvalues
    resid = ols_res.resid

    ax = axes[i][0]
    ax.scatter(df_reg[ret_col], pred, alpha=0.4, s=20, color=color)
    lims = [min(df_reg[ret_col].min(), pred.min()), max(df_reg[ret_col].max(), pred.max())]
    ax.plot(lims, lims, 'k--', lw=1.2, label='Perfect fit')
    ax.set_xlabel('Actual Monthly Return', fontsize=10)
    ax.set_ylabel('Fitted Monthly Return', fontsize=10)
    ax.set_title(f'{name}: Actual vs Fitted (R²={ols_res.rsquared:.3f})', fontsize=11, fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))

    ax2 = axes[i][1]
    ax2.plot(df_reg.index, resid, color=color, lw=1.2, alpha=0.8)
    ax2.axhline(0, color='black', lw=0.8, ls='--')
    ax2.fill_between(df_reg.index, resid, 0, alpha=0.2, color=color)
    ax2.set_title(f'{name}: Residuals Over Time', fontsize=11, fontweight='bold')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))

plt.suptitle('Regression Analysis: Macro Factors → Monthly Stock Returns', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/plot8_regression.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Coefficient Comparison ────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Apple':   ols_apple.params[1:],
    'Samsung': ols_samsung.params[1:]
}, index=['CPI Change (monthly %)', '10Y Treasury Δ (pp/month)'])

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(coef_df))
bars1 = ax.bar(x - 0.2, coef_df['Apple'],   0.35, color=APPLE_COLOR,   label='Apple', alpha=0.85)
bars2 = ax.bar(x + 0.2, coef_df['Samsung'], 0.35, color=SAMSUNG_COLOR,  label='Samsung', alpha=0.85)
for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, f'{bar.get_height():.3f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(coef_df.index, fontsize=10)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Regression Coefficient', fontsize=11)
ax.set_title('Macro Factor Coefficients: Apple vs Samsung', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('plots/plot9_coefficients.png', bbox_inches='tight')
plt.show()

**Findings — Research Question 3:**
- **Model fit is low but insightful:** The OLS models explain only ~1.6% (Apple, R²=0.016) and ~2.3% (Samsung, R²=0.023) of monthly return variance. The F-statistics are not significant at the 5% level (p > 0.2), confirming that CPI changes and treasury yield changes alone are weak predictors.
- **Direction of coefficients aligns with theory:** For Apple, higher CPI growth has a **positive** coefficient (+2.08), while rising treasury yields have a **negative** coefficient (−0.054) — consistent with theory that rate hikes harm growth stocks. Samsung shows the opposite sign for CPI (−3.36), which may reflect the Korean won/USD exchange rate dynamics and the semiconductor cycle.
- **Residuals are large:** Most of the variation in monthly returns is driven by firm-specific factors (product launches, earnings surprises, geopolitical events) rather than macro variables alone.
- **Conclusion:** Macroeconomic factors provide limited but directionally meaningful signals for both stocks. A richer model incorporating earnings momentum, market beta, and sector factors would improve explanatory power.


## 8. Conclusions & Discussion

### Summary of Findings

| Research Question | Key Finding |
|---|---|
| **RQ1: Stock Performance & Correlation** | Both companies delivered ~1000% total returns over 2015–2025. Despite being tech rivals, daily return correlation is very low (r=0.068), suggesting different market dynamics (US vs Korea exchange, sector exposure, investor base). |
| **RQ2: Financial Fundamentals** | Apple consistently outperforms on profitability margins (gross ~47% vs Samsung ~37%). Samsung's financials are more volatile, driven by semiconductor cycles. Apple's free cash flow machine (~$100B/yr) far exceeds Samsung's more capital-intensive model. |
| **RQ3: Macro Regression** | CPI and treasury yield changes explain very little of monthly return variance (R²<0.03). However, coefficient signs align with financial theory — rising rates slightly negative for Apple, mixed for Samsung. |

### Limitations
- **Currency risk**: Samsung's prices are in KRW. Our correlation and regression analysis doesn't adjust for USD/KRW exchange rate fluctuations.
- **Financial statement coverage**: Only 4–5 years of annual financials are available via yfinance, limiting longer-term trend analysis.
- **Omitted variables**: The macro regression omits important factors like market beta (S&P 500, KOSPI), earnings surprises, and sector ETF flows.
- **Survivorship bias**: Both companies are industry leaders; their performance may not generalize to the broader tech sector.
- **Samsung currency**: Direct comparison of absolute financials is difficult due to USD vs KRW denominations.

### Future Work
- Extend the regression with market-factor controls (Fama-French 3-factor model)
- Analyze the impact of product launch cycles (iPhone, Galaxy S-series) on abnormal returns
- Study the semiconductor cycle's impact on Samsung's margin volatility
- Incorporate Google Trends search data to explore consumer interest as a leading indicator


## References

1. Yahoo Finance. (2024). *Apple Inc. (AAPL) Historical Data & Financial Statements*. https://finance.yahoo.com/quote/AAPL
2. Yahoo Finance. (2024). *Samsung Electronics Co., Ltd. (005930.KS) Historical Data*. https://finance.yahoo.com/quote/005930.KS
3. Federal Reserve Bank of St. Louis (FRED). (2024). *Consumer Price Index for All Urban Consumers (CPIAUCSL)*. https://fred.stlouisfed.org/series/CPIAUCSL
4. Federal Reserve Bank of St. Louis (FRED). (2024). *10-Year Treasury Constant Maturity Rate (DGS10)*. https://fred.stlouisfed.org/series/DGS10
5. McKinney, W. (2010). Data Structures for Statistical Computing in Python. *Proceedings of the 9th Python in Science Conference*.
6. Seabold, S., & Perktold, J. (2010). Statsmodels: Econometric and Statistical Modeling with Python. *Proceedings of the 9th Python in Science Conference*.
